# 1 - Config/Libraries

In [110]:
# 🔧 Config import
import sys
import os

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(project_root)

from Config.LoggerConfig import colored_logger

logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-06-07 13:49:45] INFO - 1013015657.py - Logger initialized (Notebook)


In [111]:
from Data.Data_Fetcher import DataFetcher
import pandas as pd
import plotly.graph_objects as go

from Data.Features import LSTM_Features

from Data.Labels import LSTM_Labels

from Data.Preprocessing import Preprocessing


# 2 Data

## 2.1 - Data fetcher

### Variables

In [38]:
 #📊 Data parameters
symbol = "BTCUSDT"
start_date = "01-01-2025"
end_date = "24-04-2025"
interval = "5m"

### Process

In [39]:
# 📥 Load data

fetcher = DataFetcher(symbol, start_date, end_date, interval)

#fetcher.get_binance_data()

#fetcher.save_to_csv(directory="./Data")

fetcher.load_from_csv(directory="./Data")

fetcher.raw_data = pd.DataFrame(fetcher.raw_data, columns=['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
fetcher.raw_data = fetcher.raw_data.drop("Timestamp", axis=1)
print(fetcher.raw_data.head())

[2025-06-07 10:48:29] INFO - Data_Fetcher.py - 📥 Data loaded from: ./Data/BTCUSDT_01-01-2025_24-04-2025_5m.csv


                        Open     High      Low    Close   Volume
Timestamp                                                       
2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937
2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904
2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.385
2024-12-31 23:15:00  93687.2  93689.7  93617.7  93631.1  140.948
2024-12-31 23:20:00  93631.1  93635.2  93563.6  93607.8  151.778


### Visualization

In [41]:
print("#-----------------------")
print(fetcher.raw_data.columns)
print("#-----------------------")
print(fetcher.raw_data.head())
print("#-----------------------")
print(fetcher.raw_data.shape)
print("#-----------------------")

size = 200
df = fetcher.raw_data.copy()[-size:]

# Create a candlestick chart using Plotly
fig = go.Figure(data=[
    go.Candlestick(
        x=df.index,  # Using the timestamp index directly
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name="Candlesticks"
    )
])

# Customize layout
fig.update_layout(
    title=f"OHLC Candlestick Chart (Last {size} Points)",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=500,
    width=int(500 * 1.618)
)

# Show the chart
fig.show()

#-----------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')
#-----------------------
                        Open     High      Low    Close   Volume
Timestamp                                                       
2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937
2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904
2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.385
2024-12-31 23:15:00  93687.2  93689.7  93617.7  93631.1  140.948
2024-12-31 23:20:00  93631.1  93635.2  93563.6  93607.8  151.778
#-----------------------
(32557, 5)
#-----------------------


## 2.2 - Feature Data

### Variables

In [60]:
# Step 1: Market Sessions
resample_period = '5min'

# Step 3: Is Noise
noise_ratio = 0.15

# Step 4: Pivot Points
pivot_left = 15
pivot_right = 15

# Step 5: Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

### Process

In [45]:
# Step 0: Initialize Features Dataframe
Features_Data = LSTM_Features(fetcher.raw_data)

# Step 1: Market Sessions
Features_Data.resample_with_vwap(resample_period)

# Step 2: Market Sessions
Features_Data.market_sessions()

# Step 3: Is Noise
Features_Data.is_noise(noise_ratio)

# Step 4: Pivot Points
Features_Data.Pivot_Points(pivot_left, pivot_right)

# Step 5: Volume Pivot Points
Features_Data.Volume_Pivot_Points(duration_min, n_cross, std_factor)

[2025-06-07 10:50:05] INFO - Features.py - Starting resampling with period: 5min
[2025-06-07 10:50:05] INFO - Features.py - Basic OHLCV resampling done.
[2025-06-07 10:50:05] INFO - Features.py - VWAP calculation completed.
[2025-06-07 10:50:05] INFO - Features.py - Resampling successful. Resulting shape: (32557, 6)
[2025-06-07 10:50:05] INFO - Features.py - Starting market session flag generation.
[2025-06-07 10:50:05] INFO - Features.py - Datetime index successfully converted and localized.
[2025-06-07 10:50:06] INFO - Features.py - Market session flags successfully added.
[2025-06-07 10:50:06] INFO - Features.py - Starting noise detection with noise_ratio=0.15.
[2025-06-07 10:50:06] INFO - Features.py - Noise detection completed. Threshold used: 1.9773
[2025-06-07 10:50:06] INFO - Features.py - Starting pivot point detection with left=15, right=15.
[2025-06-07 10:50:35] INFO - Features.py - Initial pivot marking complete.
[2025-06-07 10:52:18] INFO - Features.py - Pivot filtering lo

### Visualization

In [52]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Features_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Features_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Features_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------
df = Features_Data.data.copy()[-300:]

# 1. Candlestick chart with VWAP_5m
fig1 = go.Figure()
fig1.add_trace(go.Candlestick(x=df.index,
                              open=df['Open'],
                              high=df['High'],
                              low=df['Low'],
                              close=df['Close'],
                              name="Candlestick"))
fig1.add_trace(go.Scatter(x=df.index, y=df['VWAP_5m'], mode='lines', name='VWAP_5m', line=dict(color='white')))
fig1.update_layout(title="Candlestick Chart with VWAP_5m", xaxis_title="Time", yaxis_title="Price", template="plotly_dark",xaxis_rangeslider_visible=False,)

# 2. Market session open status
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df.index, y=df["London_Open"], mode='lines', name="London"))
fig2.add_trace(go.Scatter(x=df.index, y=df["NY_Open"], mode='lines', name="New York"))
fig2.add_trace(go.Scatter(x=df.index, y=df["HK_Open"], mode='lines', name="Hong Kong"))
fig2.update_layout(title="Market Sessions", xaxis_title="Time", yaxis_title="Market Open (1=open)", template="plotly_dark")

# 3. Pivot points for price
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=df["Low"], mode='lines', name="Low", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["Low_Pivot"], mode='lines', name="Low Pivot", line=dict(width=1.2, dash='dash')))
fig3.add_trace(go.Scatter(x=df.index, y=df["High"], mode='lines', name="High", line=dict(width=0.8)))
fig3.add_trace(go.Scatter(x=df.index, y=df["High_Pivot"], mode='lines', name="High Pivot", line=dict(width=1.2, dash='dash')))
fig3.update_layout(title="Price Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

# 4. Volume pivot points
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_Low_Pivot"].tail(200), mode='lines', name="Volume Low Pivot", line=dict(width=1.2, dash='dash')))
fig4.add_trace(go.Scatter(x=df.index, y=df["VWAP_5m"].tail(200), mode='lines', name="VWAP 5m", line=dict(width=0.8)))
fig4.add_trace(go.Scatter(x=df.index, y=df["Rolling_VWAP_240min"].tail(200), mode='lines', name="Rolling VWAP 240min"))
fig4.add_trace(go.Scatter(x=df.index, y=df["Volume_High_Pivot"].tail(200), mode='lines', name="Volume High Pivot", line=dict(width=1.2, dash='dash')))
fig4.update_layout(title="Volume Pivot Points Detection", xaxis_title="Time", yaxis_title="Price", template="plotly_dark")

fig1.show()
fig2.show()
fig3.show()
fig4.show()

#----------------------------------------------------------------------
Shape: (32557, 19)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open',
       'NY_Open', 'HK_Open', 'Is_Noise', 'Dif_Low_Pivot', 'Dif_High_Pivot',
       'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot',
       'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot'],
      dtype='object')
#----------------------------------------------------------------------
                        Open     High      Low    Close   Volume  \
Timestamp                                                          
2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937   
2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904   
2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.385   
2024-12-31 23:15:00  93687.2  93689.7  93617.7  93631.1  140.948   
2024-12-31 23:20:00  93631.1 

## 2.3 - Label Data

### Variables

In [54]:
# Step 1: Categorize Pivot Points
look_forward=20

# Step 2: Categorize Volume Pivot Points
look_forward=20

### Process

In [56]:
# Step 0: Initialize Labels Dataframe
Labels_Data = LSTM_Labels(Features_Data.data)

# Step 1: Categorize Pivot Points
Labels_Data.Categorize_Pivot_Points(look_forward)

# Step 2: Categorize Volume Pivot Points
Labels_Data.Categorize_Volume_Pivot_Points(look_forward)

[2025-06-07 11:09:04] INFO - Labels.py - Starting Categorize_Pivot_Points with look_forward=20
[2025-06-07 11:09:23] INFO - Labels.py - Categorize_Pivot_Points completed successfully.
[2025-06-07 11:09:23] INFO - Labels.py - Starting Categorize_Volume_Pivot_Points with look_forward=20
[2025-06-07 11:09:39] INFO - Labels.py - Categorize_Volume_Pivot_Points completed successfully.


### Visualization

In [57]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Labels_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

df = Labels_Data.data.copy()[-300:]

# Fréquences des 0 et 1 pour les colonnes cibles
frequency = df[["Low_Below_Pivot", "High_Above_Pivot"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['High_Above_Pivot'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['Low_Below_Pivot'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Low_Pivot'
))

# Trace High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

# Fréquences des 0 et 1 pour les colonnes VWAP
frequency = df[["VWAP_Below_Volume_Low", "VWAP_Above_Volume_High"]].apply(lambda col: col.value_counts()).fillna(0).astype(int)
print("\nLabel Frequency (0 = no breakout, 1 = breakout):")
print(frequency.head(3))

# Initialiser les couleurs
colors_combined = ['lightgrey'] * len(df)
for i in range(len(df)):
    if df.iloc[i]['VWAP_Above_Volume_High'] == 1:
        colors_combined[i] = 'green'
    elif df.iloc[i]['VWAP_Below_Volume_Low'] == 1:
        colors_combined[i] = 'red'

# Création du graphique
fig = go.Figure()

# Trace VWAP_5m avec couleur dynamique
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['VWAP_5m'],
    mode='markers+lines',
    marker=dict(color=colors_combined, size=6),
    name='VWAP_5m',
    line=dict(color='lightgrey')
))

# Trace Volume_Low_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_Low_Pivot'],
    mode='lines',
    line=dict(color='red', width=1),
    name='Volume_Low_Pivot'
))

# Trace Volume_High_Pivot (ligne continue)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Volume_High_Pivot'],
    mode='lines',
    line=dict(color='green', width=1),
    name='Volume_High_Pivot'
))

# Mise en page
fig.update_layout(
    title='VWAP_5m with Volume Pivot Breakout Coloring and Pivot Lines',
    xaxis_title='Time',
    yaxis_title='Price',
    template='plotly_dark',
    height=500
)

fig.show()

#-------------------------------------------------------------------------------

#----------------------------------------------------------------------
Shape: (32557, 23)
#----------------------------------------------------------------------
Index(['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open',
       'NY_Open', 'HK_Open', 'Is_Noise', 'Dif_Low_Pivot', 'Dif_High_Pivot',
       'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot',
       'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')
#----------------------------------------------------------------------
                        Open     High      Low    Close   Volume  \
Timestamp                                                          
2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937   
2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904   
2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.

#----------------------------------------------------------------------

Label Frequency (0 = no breakout, 1 = breakout):
   VWAP_Below_Volume_Low  VWAP_Above_Volume_High
0                    220                     219
1                     80                      81


## 2.4  - Preprocessing

### Variables

In [109]:
# Step  1: Create Train/Test Data
print(Labels_Data.data.columns)
train_test_data = pd.DataFrame({"Feature_1": Labels_Data.data['VWAP_5m'],
                                "Feature_2": Labels_Data.data["London_Open"],
                                "Feature_3": Labels_Data.data["NY_Open"],
                                "Feature_4": Labels_Data.data["HK_Open"],
                                "Feature_5": Labels_Data.data["Is_Noise"],
                                "Feature_6": Labels_Data.data['Dif_Low_Pivot'],
                                "Feature_7": Labels_Data.data['Dif_High_Pivot'],
                                "Feature_8": Labels_Data.data['Dif_Volume_Low_Pivot'],
                                "Feature_9": Labels_Data.data['Dif_Volume_High_Pivot'],

                                "Label_1": Labels_Data.data['Low_Below_Pivot'],
                                "Label_2": Labels_Data.data['High_Above_Pivot'],
                                "Label_3": Labels_Data.data['VWAP_Below_Volume_Low'],
                                "Label_4": Labels_Data.data['VWAP_Above_Volume_High'],
                                }).dropna()
lookback = 50
size_test_prct = 0.3

# Step 2: Create Dataloaders
val_ratio = 0.2
feature_dim = 9
label_dim = 4
lookback = 50
reserved_ram_gb = 1.0
hidden_dim = [64, 64]

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'VWAP_5m', 'London_Open',
       'NY_Open', 'HK_Open', 'Is_Noise', 'Dif_Low_Pivot', 'Dif_High_Pivot',
       'Low_Pivot', 'High_Pivot', 'Rolling_VWAP_240min', 'Volume_Low_Pivot',
       'Volume_High_Pivot', 'Dif_Volume_Low_Pivot', 'Dif_Volume_High_Pivot',
       'Low_Below_Pivot', 'High_Above_Pivot', 'VWAP_Below_Volume_Low',
       'VWAP_Above_Volume_High'],
      dtype='object')


### Process

In [120]:
# Step 0  Initialize Processing Data
Preprocessing_Data = Preprocessing(train_test_data)

# Step 1: Create Train/Test Data
Preprocessing_Data.create_train_test_data(train_test_data, lookback,size_test_prct)

# Step 2: Suggest BatchSize
Preprocessing_Data.suggest_batch_size(feature_dim, label_dim, lookback, reserved_ram_gb, hidden_dim)

# Step 2: Create DataLoaders
train_loader, val_loader, val_dataset = Preprocessing_Data.create_dataloaders(val_ratio, hidden_dim)


[2025-06-07 15:05:38] INFO - Preprocessing.py - Starting creation of train/test data.
[2025-06-07 15:05:38] INFO - Preprocessing.py - Selected 9 feature(s) and 4 label(s).
[2025-06-07 15:05:38] INFO - Preprocessing.py - Features normalized (min-max).
[2025-06-07 15:05:38] INFO - Preprocessing.py - Generated 29336 sequences of length 50.
[2025-06-07 15:05:38] INFO - Preprocessing.py - Train/Test split: 20535 train / 8801 test samples.
[2025-06-07 15:05:38] INFO - Preprocessing.py - ✅ Train/test tensors successfully created.


TypeError: Preprocessing.suggest_batch_size() missing 2 required positional arguments: 'hidden_dim' and 'num_layers'

### Verification